In [1]:
import re

# TASK 3 - DOCUMENT CORPUS

document1 = """
University Attendance Policy

Students are expected to attend all scheduled classes regularly. Attendance is important because it helps students understand lectures and complete their coursework. The university requires students to maintain at least 75 percent attendance in each subject. Students who have less than 75 percent attendance may not be allowed to appear in the final examination.

Teachers are responsible for recording student attendance in every class. Students should check their attendance records regularly and report any mistakes to their teachers.

If a student is absent because of illness, the student should provide valid medical evidence to the department. The department may consider the medical reason, but medical evidence does not automatically remove the attendance shortage. In special cases, the department can review the student's situation and make a decision according to university rules.

Students should arrive on time and avoid unnecessary absences. Attendance records are maintained by the department throughout the semester.
"""

document2 = """
Assignment Submission Policy

Students must submit their assignments through the university learning portal before the given deadline. Every assignment has a specific submission date announced by the teacher. Students should upload their work in the required file format and check that the uploaded file is correct.

Late assignments may receive a penalty according to the course policy. Students are responsible for checking the submission status after uploading their work. If the portal shows an error, students should inform their teacher or the IT help desk.

If a student faces a serious medical emergency or a technical problem with the university portal, the teacher may allow an extension after reviewing the evidence. However, an extension is not automatic, and students must contact the teacher as soon as possible.

Teachers should clearly explain assignment requirements, deadlines, marking criteria, and accepted file formats. Students should avoid waiting until the last minute to upload their assignments.
"""

document3 = """
University IT Help Desk

The university IT help desk provides technical support to students and staff. Students can contact the help desk if they have problems with their university accounts, learning portal, email, or computer systems. The help desk normally operates from Monday to Saturday, between 9 AM and 5 PM.

Students should provide their registration number and a clear explanation of the problem when requesting support. For password problems, students can use the password recovery option on the login page. If the recovery option does not work, they should contact the IT help desk for further assistance.

The help desk usually responds to normal requests during working hours. However, urgent problems that affect online examinations or important university services may receive priority, even when the normal support schedule is busy.

Students should never share their passwords with other people. Staff members are responsible for protecting student account information and maintaining the university's computer systems.
"""

document4 = """
Updated University Attendance Policy

Students must maintain at least 80 percent attendance in each subject to be eligible for final examinations. Students with less than 80 percent attendance may not be allowed to appear in the final examination.

Teachers must record attendance in every class. Students should regularly check their attendance records and report any mistakes to their teachers.

Students with medical reasons must submit valid medical evidence to the department. The department may review special cases according to university rules.
"""

document5 = """
Assignment Extension Policy

Students must submit assignments before the announced deadline. An extension may only be granted when a student provides valid evidence of a serious medical emergency.

Technical problems with the university portal do not automatically qualify a student for an extension. Students must contact their teacher before the deadline if they face any submission problem.

The teacher is responsible for reviewing extension requests and informing students about the final decision.
"""



In [2]:
Documents = [
    document1,
    document2,
    document3,
    document4,
    document5
]

print("Total Documents:", len(Documents))

Total Documents: 5


In [3]:
# TASK 3 - LABELED QUERIES

queries = [

    # Development Queries

    {
        "query": "What is the minimum attendance required to appear in the final examination?",
        "doc": "document1",
        "evidence": "The university requires students to maintain at least 75 percent attendance in each subject.",
        "split": "dev"
    },

    {
        "query": "Can medical evidence automatically remove a student's attendance shortage?",
        "doc": "document1",
        "evidence": "medical evidence does not automatically remove the attendance shortage.",
        "split": "dev"
    },

    {
        "query": "What are the working days and hours of the university IT help desk?",
        "doc": "document3",
        "evidence": "The help desk normally operates from Monday to Saturday, between 9 AM and 5 PM.",
        "split": "dev"
    },

    {
        "query": "What should students do if the password recovery option does not work?",
        "doc": "document3",
        "evidence": "If the recovery option does not work, they should contact the IT help desk for further assistance.",
        "split": "dev"
    },

    # Testing Queries

    {
        "query": "When can a teacher allow an extension for an assignment?",
        "doc": "document2",
        "evidence": "If a student faces a serious medical emergency or a technical problem with the university portal, the teacher may allow an extension after reviewing the evidence.",
        "split": "test"
    },

    {
        "query": "What information should students provide when requesting IT support?",
        "doc": "document3",
        "evidence": "Students should provide their registration number and a clear explanation of the problem when requesting support.",
        "split": "test"
    },

    {
        "query": "What should students do if the learning portal shows an assignment submission error?",
        "doc": "document2",
        "evidence": "If the portal shows an error, students should inform their teacher or the IT help desk.",
        "split": "test"
    },

    {
        "query": "What should students do if they face a submission problem before the assignment deadline?",
        "doc": "document5",
        "evidence": "Students must contact their teacher before the deadline if they face any submission problem.",
        "split": "test"
    },

    # Unanswerable Queries

    {
        "query": "What is the university's hostel monthly fee?",
        "doc": None,
        "evidence": None,
        "split": "test"
    },

    {
        "query": "What is the university's transport fee for students?",
        "doc": None,
        "evidence": None,
        "split": "test"
    }

]

print("Total Queries:", len(queries))

Total Queries: 10


In [4]:
def word_location(text):
    return [(m.start(),m.end()) for m in re.finditer(r"\S+",text)]


In [5]:
def sentence_location(text):
    sentences = []

    for m in re.finditer(r'[^.!?]+[.!?]?', text):
        sentences.append((m.start(), m.end()))

    return sentences

In [6]:
def sentence_location(text):
    sentences = []

    for m in re.finditer(r'[^.!?]+[.!?]?', text):
        sentences.append((m.start(), m.end()))

    return sentences

In [7]:
def create_chunks(text, size=40, overlap=0):
    position = word_location(text)
    chunks = []

    if overlap < 0 or overlap >= size:
        raise ValueError("Invalid overlap")
    step = size - overlap

    for i in range(0, len(position), step):
        start = position[i][0]
        end = position[min(i + size, len(position)) - 1][1]

        chunks.append({
            "id": len(chunks) + 1,
            "start": start,
            "end": end,
            "text": text[start:end]
        })
        if i + size >= len(position):
            break

    return chunks

In [8]:
def sentence_chunks(text, budget=40):
    sentences = sentence_location(text)
    chunks = []
    current_chunk = ""
    count = 0
    chunk_start = None
    chunk_end = None

    for start, end in sentences:
        sentence = text[start:end]
        words = len(sentence.split())

        if words > budget:
            if current_chunk:
                chunks.append({
                    "id": len(chunks) + 1,
                    "start": chunk_start,
                    "end": chunk_end,
                    "text": current_chunk
                })

            current_chunk = ""
            count = 0
            chunk_start = None
            chunk_end = None

            word_spans = word_location(sentence)

            for i in range(0, len(word_spans), budget):
                s = word_spans[i][0]
                e = word_spans[min(i + budget, len(word_spans)) - 1][1]

                chunks.append({
                    "id": len(chunks) + 1,
                    "start": start + s,
                    "end": start + e,
                    "text": sentence[s:e]
                })

            continue

        chunk_start = start if chunk_start is None else chunk_start

        if count + words <= budget:
            current_chunk += sentence
            chunk_end = end
            count += words
        else:
            if current_chunk:
                chunks.append({
                    "id": len(chunks) + 1,
                    "start": chunk_start,
                    "end": chunk_end,
                    "text": current_chunk
                })

            current_chunk = sentence
            count = words
            chunk_start = start
            chunk_end = end

    if current_chunk:
        chunks.append({
            "id": len(chunks) + 1,
            "start": chunk_start,
            "end": chunk_end,
            "text": current_chunk
        })

    return chunks


In [9]:
# CHUNKING STRATEGY 1 - WORD WINDOWS (OVERLAP 0)

for i, document in enumerate(Documents, start=1):
    chunks = create_chunks(document, size=40, overlap=0)

    print(f"\nDocument {i}")
    print("Total Chunks:", len(chunks))

    for chunk in chunks:
        print(chunk["text"])



Document 1
Total Chunks: 4
University Attendance Policy

Students are expected to attend all scheduled classes regularly. Attendance is important because it helps students understand lectures and complete their coursework. The university requires students to maintain at least 75 percent attendance in each subject. Students
who have less than 75 percent attendance may not be allowed to appear in the final examination.

Teachers are responsible for recording student attendance in every class. Students should check their attendance records regularly and report any mistakes to their
teachers.

If a student is absent because of illness, the student should provide valid medical evidence to the department. The department may consider the medical reason, but medical evidence does not automatically remove the attendance shortage. In special cases, the
department can review the student's situation and make a decision according to university rules.

Students should arrive on time and avoid unnec

In [10]:
# CHUNKING STRATEGY 2 - WORD WINDOWS (OVERLAP 10)

for i, document in enumerate(Documents, start=1):
    chunks = create_chunks(document, size=40, overlap=10)

    print(f"\nDocument {i}")
    print("Total Chunks:", len(chunks))

    for chunk in chunks:
        print(chunk["text"])


Document 1
Total Chunks: 5
University Attendance Policy

Students are expected to attend all scheduled classes regularly. Attendance is important because it helps students understand lectures and complete their coursework. The university requires students to maintain at least 75 percent attendance in each subject. Students
maintain at least 75 percent attendance in each subject. Students who have less than 75 percent attendance may not be allowed to appear in the final examination.

Teachers are responsible for recording student attendance in every class. Students should check
for recording student attendance in every class. Students should check their attendance records regularly and report any mistakes to their teachers.

If a student is absent because of illness, the student should provide valid medical evidence to the department. The
student should provide valid medical evidence to the department. The department may consider the medical reason, but medical evidence does not automa

In [11]:
# CHUNKING STRATEGY 3 - SENTENCE AWARE PACKING

for i, document in enumerate(Documents, start=1):
    chunks = sentence_chunks(document, budget=40)

    print(f"\nDocument {i}")
    print("Total Chunks:", len(chunks))

    for chunk in chunks:
        print(chunk["text"])



Document 1
Total Chunks: 5

University Attendance Policy

Students are expected to attend all scheduled classes regularly. Attendance is important because it helps students understand lectures and complete their coursework. The university requires students to maintain at least 75 percent attendance in each subject.
 Students who have less than 75 percent attendance may not be allowed to appear in the final examination.

Teachers are responsible for recording student attendance in every class.
 Students should check their attendance records regularly and report any mistakes to their teachers.

If a student is absent because of illness, the student should provide valid medical evidence to the department.
 The department may consider the medical reason, but medical evidence does not automatically remove the attendance shortage. In special cases, the department can review the student's situation and make a decision according to university rules.


Students should arrive on time and avoid 

In [34]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import time

class Retriever:
    def __init__(self, records, backend="tfidf"):

        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")

        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]

        self.model_load_time = 0
        self.encoding_time = 0
        self.search_time = 0

        if backend == "tfidf":

            self.vectorizer = TfidfVectorizer()

            start_time = time.perf_counter()

            self.matrix = self.vectorizer.fit_transform(texts)

            self.encoding_time = time.perf_counter() - start_time

        elif backend == "embedding":

            from sentence_transformers import SentenceTransformer

            start_time = time.perf_counter()

            self.model = SentenceTransformer(
                "sentence-transformers/all-MiniLM-L6-v2", device="cpu")

            self.model_load_time = time.perf_counter() - start_time

            self.check_length(texts)

            start_time = time.perf_counter()

            self.matrix = self.model.encode(
                texts, normalize_embeddings=True)

            self.encoding_time = time.perf_counter() - start_time

        else:
            raise ValueError("Use tfidf or embedding")

    def check_length(self, texts):

        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)
                    ["input_ids"])

            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")

    def search(self, query, k=3, filters=None):

        start_time = time.perf_counter()

        if not query.strip() or k <= 0:
            return []

        if self.backend == "tfidf":

            q = self.vectorizer.transform([query])

            if q.nnz == 0:
                return []

            scores = (q @ self.matrix.T).toarray()[0]

        else:

            self.check_length([query])

            q = self.model.encode(
                [query], normalize_embeddings=True)[0]

            scores = self.matrix @ q

        allowed = [i for i, r in enumerate(self.records)
                   if all(r.get(key) == value
                          for key, value in (filters or {}).items())]

        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))

        results = [(self.records[i], float(scores[i]))
                   for i in ranked[:k]]

        self.search_time += time.perf_counter() - start_time

        return results

In [35]:
# STRATEGY 1 - TF-IDF + WORD WINDOWS (OVERLAP 0)

chunk_records = []

for doc_number, document in enumerate(Documents, start=1):
    chunks = create_chunks(document, size=40, overlap=0)

    for chunk in chunks:
        chunk_records.append({
            "id": f"document{doc_number}_chunk{chunk['id']}",
            "doc": f"document{doc_number}",
            "start": chunk["start"],
            "end": chunk["end"],
            "text": chunk["text"]
        })

retriever = Retriever(chunk_records, backend="tfidf")

print("\nTotal Retrieval Records:", len(chunk_records))

# TEST RETRIEVAL

test_query = queries[0]["query"]

results = retriever.search(test_query, k=3)

print("\nQuery:", test_query)

for record, score in results:
    print("\nChunk ID:", record["id"])
    print("Document:", record["doc"])
    print("Start:", record["start"])
    print("End:", record["end"])
    print("Score:", score)
    print("Text:", record["text"])


Total Retrieval Records: 17

Query: What is the minimum attendance required to appear in the final examination?

Chunk ID: document1_chunk2
Document: document1
Start: 299
End: 559
Score: 0.42136137021033854
Text: who have less than 75 percent attendance may not be allowed to appear in the final examination.

Teachers are responsible for recording student attendance in every class. Students should check their attendance records regularly and report any mistakes to their

Chunk ID: document4_chunk1
Document: document4
Start: 1
End: 263
Score: 0.41742916195164653
Text: Updated University Attendance Policy

Students must maintain at least 80 percent attendance in each subject to be eligible for final examinations. Students with less than 80 percent attendance may not be allowed to appear in the final examination.

Teachers must

Chunk ID: document1_chunk3
Document: document1
Start: 560
End: 828
Score: 0.21521463231881321
Text: teachers.

If a student is absent because of illness, the stud

In [36]:
# STRATEGY 2 - TF-IDF + WORD WINDOWS (OVERLAP 10)

chunk_records_2 = []

for doc_number, document in enumerate(Documents, start=1):
    chunks = create_chunks(document, size=40, overlap=10)

    for chunk in chunks:
        chunk_records_2.append({
            "id": f"document{doc_number}_chunk{chunk['id']}",
            "doc": f"document{doc_number}",
            "start": chunk["start"],
            "end": chunk["end"],
            "text": chunk["text"]
        })

retriever_2 = Retriever(chunk_records_2, backend="tfidf")

print("\nStrategy 2 Total Retrieval Records:", len(chunk_records_2))


Strategy 2 Total Retrieval Records: 21


In [37]:
# STRATEGY 3 - TF-IDF + SENTENCE-AWARE PACKING

chunk_records_3 = []

for doc_number, document in enumerate(Documents, start=1):
    chunks = sentence_chunks(document, budget=40)

    for chunk in chunks:
        chunk_records_3.append({
            "id": f"document{doc_number}_chunk{chunk['id']}",
            "doc": f"document{doc_number}",
            "start": chunk["start"],
            "end": chunk["end"],
            "text": chunk["text"]
        })

retriever_3 = Retriever(chunk_records_3, backend="tfidf")

print("\nStrategy 3 Total Retrieval Records:", len(chunk_records_3))


Strategy 3 Total Retrieval Records: 22


In [38]:
embedding_retriever_1 = Retriever(
    chunk_records,
    backend="embedding"
)

print("\nStrategy 1 Embedding Records:", len(chunk_records))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 920.69it/s]



Strategy 1 Embedding Records: 17


In [39]:
# STRATEGY 2 - EMBEDDING + WORD WINDOWS (OVERLAP 10)

embedding_retriever_2 = Retriever(
    chunk_records_2,
    backend="embedding"
)

print("\nStrategy 2 Embedding Records:", len(chunk_records_2))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1087.03it/s]



Strategy 2 Embedding Records: 21


In [22]:
# STRATEGY 3 - EMBEDDING + SENTENCE-AWARE PACKING

embedding_retriever_3 = Retriever(
    chunk_records_3,
    backend="embedding"
)

print("\nStrategy 3 Embedding Records:", len(chunk_records_3))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1180.34it/s]



Strategy 3 Embedding Records: 22


In [40]:
# EVALUATION OF ALL SIX CONFIGURATIONS

retrievers = {
    "TF-IDF - Overlap 0": retriever,
    "TF-IDF - Overlap 10": retriever_2,
    "TF-IDF - Sentence Aware": retriever_3,
    "Embedding - Overlap 0": embedding_retriever_1,
    "Embedding - Overlap 10": embedding_retriever_2,
    "Embedding - Sentence Aware": embedding_retriever_3
}

for strategy_name, current_retriever in retrievers.items():

    print("\n==========", strategy_name, "==========")

    for split_name in ["dev", "test"]:

        hits = 0
        reciprocal_rank = 0
        total = 0

        for item in queries:

            if item["split"] != split_name or item["doc"] is None:
                continue

            total += 1

            results = current_retriever.search(item["query"], k=3)

            rank = 0

            for position, (record, score) in enumerate(results, start=1):
                if record["doc"] == item["doc"]:
                    rank = position
                    break

            if rank > 0:
                hits += 1
                reciprocal_rank += 1 / rank

        print("\n", split_name.upper())
        print("Total Queries:", total)
        print("Hit@3:", hits / total)
        print("MRR@3:", reciprocal_rank / total)


========== TF-IDF - Overlap 0 ==========

 DEV
Total Queries: 4
Hit@3: 1.0
MRR@3: 1.0

 TEST
Total Queries: 4
Hit@3: 1.0
MRR@3: 1.0

========== TF-IDF - Overlap 10 ==========

 DEV
Total Queries: 4
Hit@3: 1.0
MRR@3: 0.8333333333333333

 TEST
Total Queries: 4
Hit@3: 1.0
MRR@3: 1.0

========== TF-IDF - Sentence Aware ==========

 DEV
Total Queries: 4
Hit@3: 1.0
MRR@3: 1.0

 TEST
Total Queries: 4
Hit@3: 1.0
MRR@3: 1.0

========== Embedding - Overlap 0 ==========

 DEV
Total Queries: 4
Hit@3: 1.0
MRR@3: 0.875

 TEST
Total Queries: 4
Hit@3: 0.75
MRR@3: 0.41666666666666663

========== Embedding - Overlap 10 ==========

 DEV
Total Queries: 4
Hit@3: 1.0
MRR@3: 1.0

 TEST
Total Queries: 4
Hit@3: 0.5
MRR@3: 0.3333333333333333

========== Embedding - Sentence Aware ==========

 DEV
Total Queries: 4
Hit@3: 1.0
MRR@3: 0.875

 TEST
Total Queries: 4
Hit@3: 0.75
MRR@3: 0.5833333333333333


In [29]:
# MODEL TOKEN LENGTH AUDIT

for name, current_retriever in embedding_retrievers.items():

    lengths = [
        len(current_retriever.model.tokenizer(
            record["text"], truncation=False
        )["input_ids"])
        for record in current_retriever.records
    ]

    print("\n", name)
    print("Total Chunks:", len(lengths))
    print("Maximum Token Length:", max(lengths))
    print("Model Token Limit:", current_retriever.model.max_seq_length)


 Strategy 1 - Overlap 0
Total Chunks: 17
Maximum Token Length: 48
Model Token Limit: 256

 Strategy 2 - Overlap 10
Total Chunks: 21
Maximum Token Length: 49
Model Token Limit: 256

 Strategy 3 - Sentence Aware
Total Chunks: 22
Maximum Token Length: 45
Model Token Limit: 256


In [41]:
# FINAL RETRIEVAL RESULTS AND CSV LOG

import csv

retrievers = {
    "TF-IDF - Overlap 0": retriever,
    "TF-IDF - Overlap 10": retriever_2,
    "TF-IDF - Sentence Aware": retriever_3,
    "Embedding - Overlap 0": embedding_retriever_1,
    "Embedding - Overlap 10": embedding_retriever_2,
    "Embedding - Sentence Aware": embedding_retriever_3
}

csv_file = open("retrieval_log.csv", "w", newline="", encoding="utf-8")
writer = csv.writer(csv_file)

writer.writerow([
    "Strategy", "Query", "Split", "Rank",
    "Chunk ID", "Document", "Score",
    "Start", "End", "Text"
])

failures = []

for strategy_name, current_retriever in retrievers.items():

    print("\n==========", strategy_name, "==========")

    for split_name in ["dev", "test"]:

        hits = 0
        reciprocal_rank = 0
        total = 0

        for item in queries:

            if item["split"] != split_name or item["doc"] is None:
                continue

            total += 1
            results = current_retriever.search(item["query"], k=3)

            rank = 0

            for position, (record, score) in enumerate(results, start=1):

                if record["doc"] == item["doc"]:
                    rank = position
                    break

            if rank > 0:
                hits += 1
                reciprocal_rank += 1 / rank

            elif split_name == "test":
                failures.append((strategy_name, item["query"]))

        print("\n", split_name.upper())
        print("Total Queries:", total)
        print("Hit@3:", hits / total)
        print("MRR@3:", reciprocal_rank / total)

    for item in queries:

        results = current_retriever.search(item["query"], k=3)

        print("\nQuery:", item["query"])

        for position, (record, score) in enumerate(results, start=1):

            print("\nRank:", position)
            print("Chunk ID:", record["id"])
            print("Document:", record["doc"])
            print("Score:", score)
            print("Start:", record["start"])
            print("End:", record["end"])
            print("Text:", record["text"])

            writer.writerow([
                strategy_name, item["query"], item["split"],
                position, record["id"], record["doc"],
                score, record["start"], record["end"], record["text"]
            ])

    print("\nChunk Count:", len(current_retriever.records))
    print("Model Loading Time:", current_retriever.model_load_time)
    print("Encoding Time:", current_retriever.encoding_time)
    print("Search Time:", current_retriever.search_time)

csv_file.close()

print("\nCSV file exported: retrieval_log.csv")

print("\n========== RETRIEVAL FAILURES ==========")

for strategy_name, query in failures:
    print(strategy_name, ":", query)

print("\nTotal Failures:", len(failures))


========== TF-IDF - Overlap 0 ==========

 DEV
Total Queries: 4
Hit@3: 1.0
MRR@3: 1.0

 TEST
Total Queries: 4
Hit@3: 1.0
MRR@3: 1.0

Query: What is the minimum attendance required to appear in the final examination?

Rank: 1
Chunk ID: document1_chunk2
Document: document1
Score: 0.42136137021033854
Start: 299
End: 559
Text: who have less than 75 percent attendance may not be allowed to appear in the final examination.

Teachers are responsible for recording student attendance in every class. Students should check their attendance records regularly and report any mistakes to their

Rank: 2
Chunk ID: document4_chunk1
Document: document4
Score: 0.41742916195164653
Start: 1
End: 263
Text: Updated University Attendance Policy

Students must maintain at least 80 percent attendance in each subject to be eligible for final examinations. Students with less than 80 percent attendance may not be allowed to appear in the final examination.

Teachers must

Rank: 3
Chunk ID: document1_chunk3
Documen

AttributeError: 'Retriever' object has no attribute 'model_load_time'